In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm  
import nltk
#nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize
import json
from pathlib import Path

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('ProsusAI/finbert')
base = Path('/Users/dylan/Desktop/Research-Project/Earning-calls-sentiment/data/Raw/transcripts/A')

with open(base / '2006Q1.json') as f:
    transcript1 = json.load(f)
with open(base / '2006Q2.json') as f:
    transcript2 = json.load(f)

In [ ]:
text1 = transcript1[0]['content']
sentences1 = sent_tokenize(text1)
text2 = transcript2[0]['content']
sentences2 = sent_tokenize(text2)

In [ ]:
print(f"Transcript 1: {len(sentences1)} sentences")
print(f"Transcript 2: {len(sentences2)} sentences")
print(f"\nFirst 3 sentences of transcript 1:")
for s in sentences1[:3]:
    print(f"  - {s}")

In [ ]:
encoded1 = tokenizer(sentences1, padding=True, truncation=True, return_tensors='pt')
encoded2 = tokenizer(sentences2, padding=True, truncation=True, return_tensors='pt')

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained('ProsusAI/finbert')
model.eval()

In [ ]:
def get_FinBERT_sent(encoded):
    with torch.no_grad():
        outputs = model(**encoded)
        
    probs = torch.softmax(outputs.logits, dim=-1)
    
    predicted_class_ids = probs.argmax(dim=-1)
    predicted_labels = [model.config.id2label[i.item()] for i in predicted_class_ids]

    results = pd.DataFrame({
        'predicted': predicted_labels,
        'positive': probs[:, 0].numpy(),
        'negative': probs[:, 1].numpy(),
        'neutral':  probs[:, 2].numpy(),
    })

    return results

In [ ]:
sentiment1 = get_FinBERT_sent(encoded1)
sentiment2 = get_FinBERT_sent(encoded2)

In [ ]:
sentiment1.head()

In [ ]:
sentiment2.head()